The **Shannon Diversity Index** (from information theory / ecology) instead measures the *entropy* of the supplier distribution:

$$H = -\sum_{i=1}^{n} p_i \ln(p_i)$$

where $p_i$ is partner $i$'s share of imports.

| Property | HHI | Shannon H |
|---|---|---|
| Range | [1/n, 1] | [0, ln(n)] |
| Sensitivity | Emphasizes large shares (squared) | Sensitive across the full distribution |
| Interpretation | Higher = more concentrated | Higher = more diverse |
| Effective suppliers | 1/HHI | e^H |

Shannon catches changes in the *middle* of the distribution that HHI misses. Two countries can have identical HHI but different Shannon values if their smaller suppliers are distributed differently. This matters for food security: having 5 small backup suppliers vs 1 is invisible to HHI but real in a crisis.

---

## 0. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# ---- Paths (relative to repo root) ----
# Adjust ROOT if running from a different working directory.
ROOT = Path(".").resolve().parent  # assumes notebook is in notebooks/
TRADE_MATRIX = ROOT / "data" / "cleaned" / "trade_matrix_cleaned.csv"
FBS_PATH = ROOT / "data" / "cleaned" / "fbs_cleaned.csv"
OUTPUT_DIR = ROOT / "data" / "cleaned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Item-to-commodity mapping (matches build_quantity_concentration.py)
ITEM_MAP = {
    "Wheat": "Wheat",
    "Wheat and meslin flour": "Wheat",
    "Rice, paddy (rice milled equivalent)": "Rice",
    "Rice, milled": "Rice",
    "Maize (corn)": "Maize",
}

COMMODITIES = ["Wheat", "Rice", "Maize"]

print(f"Trade matrix path: {TRADE_MATRIX}")
print(f"Trade matrix exists: {TRADE_MATRIX.exists()}")
print(f"FBS path: {FBS_PATH}")
print(f"FBS exists: {FBS_PATH.exists()}")

## 1. Load & Prepare Trade Data

We read directly from `trade_matrix_cleaned.csv` (the committed, analysis-ready bilateral trade file) rather than intermediate parquets, because those are gitignored and may not exist on every machine.

In [ ]:
raw = pd.read_csv(TRADE_MATRIX)
print(f"Loaded {len(raw):,} rows, columns: {raw.columns.tolist()}")
raw.head(3)

In [ ]:
# Filter: import quantities in tonnes, positive values only
# (mirrors build_quantity_concentration.py logic exactly)
df = raw[
    (raw["Element"] == "Import quantity")
    & (raw["Unit"] == "t")
    & (raw["Value"] > 0)
].copy()

df["Commodity"] = df["Item"].map(ITEM_MAP)
df = df[df["Commodity"].notna()].copy()

print(f"After filtering to staple import quantities: {len(df):,} rows")
print(f"Commodities: {df['Commodity'].value_counts().to_dict()}")
print(f"Years: {df['Year'].min()} - {df['Year'].max()}")
print(f"Reporters: {df['Reporter Countries'].nunique()}")

## 2. Compute Partner Shares

Aggregate detailed items (e.g. "Wheat" + "Wheat and meslin flour") into the 3 commodity groups, then compute each partner's share of a reporter's total imports for that commodity-year.

In [ ]:
GROUP_KEYS = [
    "Reporter Country Code", "Reporter Countries",
    "Commodity", "Year",
]

FLOW_KEYS = GROUP_KEYS + [
    "Partner Country Code", "Partner Countries",
]

# Aggregate flows: sum across detailed items within commodity group
flows = (
    df.groupby(FLOW_KEYS, as_index=False)["Value"]
    .sum()
    .rename(columns={"Value": "import_quantity_t"})
)

# Total imports per reporter-commodity-year
totals = (
    flows.groupby(GROUP_KEYS, as_index=False)["import_quantity_t"]
    .sum()
    .rename(columns={"import_quantity_t": "total_import_quantity_t"})
)

flows = flows.merge(totals, on=GROUP_KEYS, how="left")
flows["partner_share"] = flows["import_quantity_t"] / flows["total_import_quantity_t"]

print(f"Flow-level rows: {len(flows):,}")
print(f"Reporter-commodity-year groups: {len(totals):,}")
flows[["Reporter Countries", "Partner Countries", "Commodity", "Year",
       "import_quantity_t", "total_import_quantity_t", "partner_share"]].head(8)

## 3. Shannon Diversity Index Implementation

$$H = -\sum_{i=1}^{n} p_i \ln(p_i)$$

**Properties:**
- $H = 0$ when a country has a single supplier (zero diversity).
- $H = \ln(n)$ when all $n$ suppliers contribute equally (maximum diversity).
- **Effective suppliers (Shannon):** $e^H$ — directly comparable to the HHI-based $1/\text{HHI}$.

We also compute the **Shannon Evenness Index** $E = H / \ln(n)$, which normalizes to [0, 1] regardless of partner count. $E = 1$ means perfectly even distribution; $E \to 0$ means dominated by one partner.

In [ ]:
def shannon_diversity(shares: pd.Series) -> float:
    """Compute Shannon Diversity Index H = -sum(p * ln(p)).

    Filters out zero/negative shares to avoid log(0).
    Returns 0.0 for single-supplier cases.
    """
    p = shares[shares > 0].values
    if len(p) <= 1:
        return 0.0
    return -np.sum(p * np.log(p))


def shannon_evenness(h: float, n: int) -> float:
    """Compute Shannon Evenness E = H / ln(n).

    Returns NaN when n <= 1 (evenness undefined for single supplier).
    """
    if n <= 1:
        return np.nan
    return h / np.log(n)


# --- Compute SDI per reporter-commodity-year ---
sdi = (
    flows.groupby(GROUP_KEYS, as_index=False)
    .agg(
        shannon_h=("partner_share", shannon_diversity),
        partner_count=("Partner Country Code", "nunique"),
    )
)

# Effective suppliers (Shannon-based): e^H
sdi["effective_suppliers_shannon"] = np.exp(sdi["shannon_h"])

# Shannon evenness: H / ln(n)
sdi["shannon_evenness"] = sdi.apply(
    lambda row: shannon_evenness(row["shannon_h"], row["partner_count"]),
    axis=1,
)

print(f"Shannon metrics computed for {len(sdi):,} reporter-commodity-year groups")
sdi.describe().round(4)

## 4. Compute HHI (Matching Existing Pipeline)

We recompute HHI here so that both indices live side-by-side in a single output. This matches `build_quantity_concentration.py` exactly.

In [ ]:
# --- HHI ---
hhi = (
    flows.assign(share_sq=flows["partner_share"] ** 2)
    .groupby(GROUP_KEYS, as_index=False)["share_sq"]
    .sum()
    .rename(columns={"share_sq": "partner_hhi"})
)

# Effective suppliers (HHI-based): 1 / HHI
hhi["effective_suppliers_hhi"] = 1.0 / hhi["partner_hhi"]

# --- Additional concentration metrics ---
other_metrics = (
    flows.groupby(GROUP_KEYS, as_index=False)
    .agg(
        top_partner_share=("partner_share", "max"),
        suppliers_over_5pct=("partner_share", lambda x: (x > 0.05).sum()),
        total_import_quantity_t=("import_quantity_t", "sum"),
    )
)

# --- Dominant partner ---
flows["rank"] = flows.groupby(GROUP_KEYS)["partner_share"].rank(
    method="first", ascending=False
)
top_partner = (
    flows[flows["rank"] == 1][GROUP_KEYS + ["Partner Country Code", "Partner Countries"]]
    .rename(columns={
        "Partner Country Code": "dominant_partner_code",
        "Partner Countries": "dominant_partner",
    })
)

print(f"HHI computed for {len(hhi):,} groups")
hhi["partner_hhi"].describe().round(4)

## 5. Combine All Metrics & Save to Cleaned Data

This is the key step the project was missing: a single, stored output containing **both** HHI and Shannon metrics that downstream scripts can reference.

Output: `data/cleaned/concentration_with_shannon.csv`

In [ ]:
# Merge all metrics into one table
concentration = (
    sdi
    .merge(hhi, on=GROUP_KEYS, how="left")
    .merge(other_metrics, on=GROUP_KEYS, how="left")
    .merge(top_partner, on=GROUP_KEYS, how="left")
)

# Order columns for readability
concentration = concentration[
    [
        "Reporter Country Code", "Reporter Countries",
        "Year", "Commodity",
        "partner_count", "total_import_quantity_t",
        # HHI family
        "partner_hhi", "effective_suppliers_hhi",
        # Shannon family
        "shannon_h", "effective_suppliers_shannon", "shannon_evenness",
        # Other
        "top_partner_share", "suppliers_over_5pct",
        "dominant_partner_code", "dominant_partner",
    ]
].sort_values(["Reporter Country Code", "Year", "Commodity"])

# Save to cleaned data
out_path = OUTPUT_DIR / "concentration_with_shannon.csv"
concentration.to_csv(out_path, index=False)
print(f"Saved {len(concentration):,} rows to {out_path}")
print(f"Countries: {concentration['Reporter Country Code'].nunique()}")
print(f"Years: {concentration['Year'].min()} - {concentration['Year'].max()}")
print(f"Commodities: {concentration['Commodity'].unique().tolist()}")
concentration.head(10)

## 6. Shannon vs HHI — Comparative Analysis

How do the two indices relate? Where do they agree, and where does Shannon reveal something HHI misses?

In [ ]:
# --- 6a. Correlation ---
# Shannon H is a diversity measure (higher = more diverse)
# HHI is a concentration measure (higher = more concentrated)
# So we expect a strong NEGATIVE correlation.

corr = concentration[["partner_hhi", "shannon_h", "shannon_evenness",
                       "effective_suppliers_hhi", "effective_suppliers_shannon",
                       "top_partner_share", "partner_count"]].corr()
print("Correlation matrix (all commodity-country-years):")
print(corr.round(3))

In [ ]:
# --- 6b. Scatter: HHI vs Shannon H, coloured by commodity ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, commodity in zip(axes, COMMODITIES):
    subset = concentration[concentration["Commodity"] == commodity]
    ax.scatter(
        subset["partner_hhi"], subset["shannon_h"],
        alpha=0.3, s=15, edgecolors="none",
    )
    ax.set_xlabel("HHI (concentration)")
    ax.set_title(commodity)
    r = subset[["partner_hhi", "shannon_h"]].corr().iloc[0, 1]
    ax.annotate(f"r = {r:.3f}", xy=(0.05, 0.95), xycoords="axes fraction",
                fontsize=11, va="top")

axes[0].set_ylabel("Shannon H (diversity)")
fig.suptitle("HHI vs Shannon Diversity Index by Commodity", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(ROOT / "visualizations" / "hhi_vs_shannon_scatter.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 6c. Effective suppliers comparison ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(
    concentration["effective_suppliers_hhi"],
    concentration["effective_suppliers_shannon"],
    alpha=0.2, s=10, edgecolors="none",
)
# 45-degree reference line
max_val = max(
    concentration["effective_suppliers_hhi"].max(),
    concentration["effective_suppliers_shannon"].max(),
)
ax.plot([0, max_val], [0, max_val], "--", color="grey", linewidth=0.8, label="y = x")
ax.set_xlabel("Effective suppliers (1/HHI)")
ax.set_ylabel("Effective suppliers (e^H)")
ax.set_title("Effective Suppliers: HHI vs Shannon")
ax.legend()
plt.tight_layout()
plt.savefig(str(ROOT / "visualizations" / "effective_suppliers_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print("\nWhere Shannon and HHI diverge most (Shannon sees more effective suppliers):")
concentration["eff_diff"] = (
    concentration["effective_suppliers_shannon"] - concentration["effective_suppliers_hhi"]
)
divergent = concentration.nlargest(15, "eff_diff")[
    ["Reporter Countries", "Commodity", "Year",
     "partner_count", "partner_hhi", "shannon_h",
     "effective_suppliers_hhi", "effective_suppliers_shannon", "eff_diff"]
]
print(divergent.to_string(index=False))

In [ ]:
# --- 6d. Shannon Evenness distribution ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, commodity in zip(axes, COMMODITIES):
    subset = concentration[
        (concentration["Commodity"] == commodity)
        & concentration["shannon_evenness"].notna()
    ]
    ax.hist(subset["shannon_evenness"], bins=40, edgecolor="white", alpha=0.8)
    ax.axvline(subset["shannon_evenness"].median(), color="red",
               linestyle="--", label=f"median={subset['shannon_evenness'].median():.2f}")
    ax.set_xlabel("Shannon Evenness (0=dominated, 1=equal)")
    ax.set_title(commodity)
    ax.legend()

axes[0].set_ylabel("Count")
fig.suptitle("Distribution of Shannon Evenness by Commodity", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(ROOT / "visualizations" / "shannon_evenness_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 7. Shannon Diversity Over Time — Trend Analysis

How has supplier diversity evolved? In particular, did the 2022 Black Sea disruption cause lasting diversification or temporary blips?

In [ ]:
# --- 7a. Global median Shannon H per commodity-year ---
yearly_median = (
    concentration.groupby(["Year", "Commodity"], as_index=False)
    .agg(
        median_shannon_h=("shannon_h", "median"),
        median_hhi=("partner_hhi", "median"),
        median_evenness=("shannon_evenness", "median"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for commodity in COMMODITIES:
    subset = yearly_median[yearly_median["Commodity"] == commodity]
    axes[0].plot(subset["Year"], subset["median_shannon_h"], marker="o",
                 markersize=4, label=commodity)
    axes[1].plot(subset["Year"], subset["median_hhi"], marker="o",
                 markersize=4, label=commodity)

axes[0].set_ylabel("Median Shannon H")
axes[0].set_title("Supplier Diversity Over Time (Shannon)")
axes[0].legend()
axes[1].set_ylabel("Median HHI")
axes[1].set_title("Supplier Concentration Over Time (HHI)")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("Year")
    ax.axvline(2022, color="red", linestyle=":", alpha=0.5, label="2022 disruption")

plt.tight_layout()
plt.savefig(str(ROOT / "visualizations" / "diversity_trends_over_time.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 7b. Country-level Shannon trends for key importers ---
# These are the countries highlighted in the data audit as interesting cases.
spotlight_countries = [
    "Egypt", "Türkiye", "Saudi Arabia", "Japan",
    "Indonesia", "Philippines", "Republic of Korea", "Mexico",
]

spotlight = concentration[
    concentration["Reporter Countries"].isin(spotlight_countries)
    & (concentration["Commodity"] == "Wheat")
].copy()

if not spotlight.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for country in spotlight["Reporter Countries"].unique():
        sub = spotlight[spotlight["Reporter Countries"] == country]
        axes[0].plot(sub["Year"], sub["shannon_h"], marker=".", label=country)
        axes[1].plot(sub["Year"], sub["effective_suppliers_shannon"],
                     marker=".", label=country)

    axes[0].set_ylabel("Shannon H")
    axes[0].set_title("Wheat: Shannon Diversity by Importer")
    axes[1].set_ylabel("Effective Suppliers (e^H)")
    axes[1].set_title("Wheat: Effective Suppliers (Shannon)")

    for ax in axes:
        ax.set_xlabel("Year")
        ax.legend(fontsize=8, ncol=2)
        ax.axvline(2022, color="red", linestyle=":", alpha=0.5)

    plt.tight_layout()
    plt.savefig(str(ROOT / "visualizations" / "wheat_shannon_spotlight_countries.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Spotlight countries not found in data — check country names.")

## 8. Lowest-Diversity Countries (Risk Flag)

Which countries have the least supplier diversity for each commodity in the most recent 3-year window?

In [ ]:
# Use the latest 3-year window available in the data
latest_year = concentration["Year"].max()
window = concentration[concentration["Year"].between(latest_year - 2, latest_year)].copy()

# Average across the 3-year window
window_avg = (
    window.groupby(["Reporter Country Code", "Reporter Countries", "Commodity"],
                   as_index=False)
    .agg(
        mean_shannon_h=("shannon_h", "mean"),
        mean_hhi=("partner_hhi", "mean"),
        mean_evenness=("shannon_evenness", "mean"),
        mean_eff_suppliers_shannon=("effective_suppliers_shannon", "mean"),
        mean_top_partner_share=("top_partner_share", "mean"),
        mean_import_t=("total_import_quantity_t", "mean"),
    )
)

print(f"Window: {latest_year - 2}-{latest_year}")
print(f"Country-commodity groups: {len(window_avg)}\n")

for commodity in COMMODITIES:
    sub = window_avg[window_avg["Commodity"] == commodity]
    # Filter to meaningful importers (at least 10,000 t/year average)
    significant = sub[sub["mean_import_t"] >= 10_000]
    bottom = significant.nsmallest(10, "mean_shannon_h")
    print(f"\n--- {commodity}: Lowest Shannon Diversity (significant importers, >=10kt/yr) ---")
    print(bottom[
        ["Reporter Countries", "mean_shannon_h", "mean_hhi", "mean_evenness",
         "mean_eff_suppliers_shannon", "mean_top_partner_share", "mean_import_t"]
    ].to_string(index=False, float_format="{:.3f}".format))

## 9. Quick Validation Checks

In [ ]:
# --- Sanity checks ---

print("=" * 50)
print("VALIDATION")
print("=" * 50)

# 1. Shannon H should be 0 when partner_count == 1
single = concentration[concentration["partner_count"] == 1]
assert (single["shannon_h"] == 0.0).all(), "Shannon H should be 0 for single-supplier cases"
print(f"[PASS] Single-supplier cases: {len(single)} rows, all have H=0")

# 2. Shannon H <= ln(partner_count) always
max_h = np.log(concentration["partner_count"])
violations = concentration["shannon_h"] > max_h + 1e-10
assert not violations.any(), "Shannon H exceeds theoretical maximum"
print(f"[PASS] Shannon H <= ln(n) for all {len(concentration)} rows")

# 3. HHI and Shannon should be negatively correlated
r = concentration[["partner_hhi", "shannon_h"]].corr().iloc[0, 1]
assert r < -0.5, f"Expected strong negative HHI-Shannon correlation, got r={r:.3f}"
print(f"[PASS] HHI-Shannon correlation: r={r:.3f} (strong negative as expected)")

# 4. Evenness in [0, 1] where defined
valid_e = concentration["shannon_evenness"].dropna()
assert valid_e.between(0, 1 + 1e-10).all(), "Evenness out of [0,1] range"
print(f"[PASS] Shannon Evenness in [0,1] for all {len(valid_e)} non-null rows")

# 5. Effective suppliers (Shannon) >= 1 always
assert (concentration["effective_suppliers_shannon"] >= 1.0 - 1e-10).all()
print(f"[PASS] Effective suppliers (Shannon) >= 1 for all rows")

# 6. Output file exists and is non-empty
out_check = pd.read_csv(out_path, nrows=5)
assert "shannon_h" in out_check.columns, "shannon_h missing from output"
assert "partner_hhi" in out_check.columns, "partner_hhi missing from output"
print(f"[PASS] Output file {out_path.name} has both HHI and Shannon columns")

print("\nAll checks passed.")

## 9. Output Summary

In [ ]:
print("=" * 55)
print("OUTPUT SUMMARY")
print("=" * 55)
print(f"\nFile: {out_path}")
print(f"Rows: {len(concentration):,}")
print(f"Countries: {concentration['Reporter Country Code'].nunique()}")
print(f"Years: {concentration['Year'].min()} - {concentration['Year'].max()}")
print(f"Commodities: {concentration['Commodity'].unique().tolist()}")
print(f"\nColumns:")
for col in concentration.columns:
    dtype = concentration[col].dtype
    nulls = concentration[col].isna().sum()
    print(f"  {col:<35} {str(dtype):<10} nulls={nulls}")

print(f"\nDescriptive statistics (key metrics):")
print(
    concentration[
        ["partner_hhi", "shannon_h", "shannon_evenness",
         "effective_suppliers_hhi", "effective_suppliers_shannon",
         "top_partner_share", "partner_count"]
    ].describe().round(4).to_string()
)

print("\nVisualizations saved to: visualizations/")
print("  - hhi_vs_shannon_scatter.png")
print("  - effective_suppliers_comparison.png")
print("  - shannon_evenness_distribution.png")
print("  - diversity_trends_over_time.png")
print("  - wheat_shannon_spotlight_countries.png")